# 深度学习 HW04

学号：20234080324  
姓名：李英华

In [1]:
import math
import re
from collections import OrderedDict

import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(42)

## 2 语言模型

### 2.1 理论题

语料为 `ababc`，bigram 为

$$
(a,b),(b,a),(a,b),(b,c).
$$

在上一词为 `b` 时，$C(b,a)=1,C(b,c)=1,C(b)=2$，词表大小 $V=3$。加一平滑为

$$
p(w|b)=\frac{C(b,w)+1}{C(b)+V}.
$$

因此

$$
p(a|b)=\frac{1+1}{2+3}=0.4,
\qquad
p(c|b)=\frac{1+1}{2+3}=0.4.
$$

### 2.2 编程题

实现 `preprocess_text(text, n)`，完成小写化、分词、词到整数 ID 的映射，并生成长度为 `n` 的特征和下一个词标签。

In [2]:
def preprocess_text(text, n):
    text = text.lower()
    tokens = re.findall(r"[a-z0-9']+|[\u4e00-\u9fff]", text)

    token_to_id = OrderedDict()
    for token in tokens:
        if token not in token_to_id:
            token_to_id[token] = len(token_to_id)

    features, labels = [], []
    for i in range(max(0, len(tokens) - n + 1)):
        features.append(tokens[i:i + n])
        labels.append(tokens[i + n] if i + n < len(tokens) else None)

    ids = [token_to_id[token] for token in tokens]
    return tokens, dict(token_to_id), ids, features, labels


tokens, token_to_id, ids, features, labels = preprocess_text("The time machine", 2)
print("tokens =", tokens)
print("token_to_id =", token_to_id)
print("ids =", ids)
print("features =", features)
print("labels =", labels)

tokens = ['the', 'time', 'machine']
token_to_id = {'the': 0, 'time': 1, 'machine': 2}
ids = [0, 1, 2]
features = [['the', 'time'], ['time', 'machine']]
labels = ['machine', None]


## 3 RNN

### 3.1 理论题

线性 RNN 为

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t,
\qquad
o_t=W_{oh}h_t.
$$

损失函数为

$$
L=\frac12\sum_{t=1}^{T}\lVert o_t-y_t\rVert^2.
$$

令 $e_t=o_t-y_t$，$\delta_t=\partial L/\partial h_t$，则 BPTT 递推为

$$
\delta_t=W_{oh}^Te_t+W_{hh}^T\delta_{t+1},
\qquad
\delta_{T+1}=0.
$$

所以

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\delta_t h_{t-1}^T.
$$

由于反向传播中反复乘以 $W_{hh}^T$，当其特征值较大或较小时，容易出现梯度爆炸或梯度消失。

### 3.2 编程题

实现单步 RNN 的前向传播和反向传播，激活函数使用 `tanh`。

In [3]:
def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    da = dh_next * (1 - h_t ** 2)
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(axis=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


rng = np.random.default_rng(42)
x_t = rng.normal(size=(4, 3))
h_prev = rng.normal(size=(4, 5))
W_hx = rng.normal(size=(3, 5))
W_hh = rng.normal(size=(5, 5))
b_h = rng.normal(size=(5,))
dh_next = rng.normal(size=(4, 5))

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
grads = rnn_step_backward(dh_next, cache)
print("h_t shape =", h_t.shape)
print("dx_t shape =", grads[0].shape)
print("dh_prev shape =", grads[1].shape)
print("dW_hx shape =", grads[2].shape)
print("dW_hh shape =", grads[3].shape)
print("db_h shape =", grads[4].shape)

h_t shape = (4, 5)
dx_t shape = (4, 3)
dh_prev shape = (4, 5)
dW_hx shape = (3, 5)
dW_hh shape = (5, 5)
db_h shape = (5,)


## 4 双向 RNN

### 4.1 理论题

设双向 RNN 有 $L$ 层，每个方向的隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$。

第 1 层每个方向参数量为 $DH+H^2+H$，双向为

$$
2(DH+H^2+H).
$$

从第 2 层开始，输入维度变为 $2H$，每层双向参数量为

$$
2(2H\cdot H+H^2+H)=6H^2+2H.
$$

输出层参数量为 $2HO+O$，所以总参数量为

$$
2(DH+H^2+H)+(L-1)(6H^2+2H)+2HO+O.
$$

### 4.2 编程题

使用 `torch.nn.RNN` 实现双向 RNN，输入形状为 `(seq_len, batch, input_dim)`，输出形状为 `(seq_len, batch, 2*hidden_dim)`。

In [4]:
class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        seq_repr = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        return outputs, seq_repr


X = torch.randn(5, 3, 4)
model = BiRNNEncoder(input_dim=4, hidden_dim=6)
outputs, seq_repr = model(X)
print("outputs shape =", outputs.shape)
print("sequence representation shape =", seq_repr.shape)

outputs shape = torch.Size([5, 3, 12])
sequence representation shape = torch.Size([3, 12])


## 5 词嵌入向量

### 5.1 理论题

Skip-gram 负采样中，中心词向量为 $v_c$，正样本输出向量为 $u_o$，负样本输出向量为 $u_{n_k}$。最大化目标为

$$
\log\sigma(u_o^Tv_c)+\sum_{k=1}^{K}\log\sigma(-u_{n_k}^Tv_c).
$$

因此损失函数为

$$
L=-\log\sigma(u_o^Tv_c)-\sum_{k=1}^{K}\log\sigma(-u_{n_k}^Tv_c).
$$

### 5.2 编程题

实现 CBOW 的完整 softmax 损失。输入权重矩阵 $W\in\mathbb{R}^{V\times d}$，输出权重矩阵 $W_{out}\in\mathbb{R}^{d\times V}$。

In [5]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=-1, keepdims=True)


def cbow_loss(context_ids, target_id, W, W_out):
    h = W[context_ids].mean(axis=0)
    logits = h @ W_out
    probs = softmax(logits)
    loss = -np.log(probs[target_id] + 1e-12)
    return loss, probs


rng = np.random.default_rng(42)
V, d = 8, 4
W = rng.normal(size=(V, d))
W_out = rng.normal(size=(d, V))
loss, probs = cbow_loss([0, 2, 4], 3, W, W_out)
print("loss =", loss)
print("sum(probs) =", probs.sum())

loss = 1.0010474948711128
sum(probs) = 0.9999999999999999


## 6 注意力机制

### 6.1 理论题

给定 $Q\in\mathbb{R}^{2\times4}$，$K\in\mathbb{R}^{3\times4}$，$V\in\mathbb{R}^{3\times5}$，且 $d_k=4$。缩放点积注意力为

$$
score=\frac{QK^T}{\sqrt{d_k}}=\frac{QK^T}{2}.
$$

其中 $QK^T$ 的形状为 $2\times3$，softmax 后仍为 $2\times3$。最终输出为

$$
Attention(Q,K,V)=softmax(score)V,
$$

形状为

$$
(2\times3)(3\times5)=2\times5.
$$

### 6.2 编程题

实现多头注意力。设 `num_heads=2`，`d_model=4`，输入 `X` 的形状为 `(seq_len, batch, d_model)`，输出形状与输入相同。

In [6]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, X):
        seq_len, batch, d_model = X.shape
        X = X.view(seq_len, batch, self.num_heads, self.d_k)
        return X.permute(1, 2, 0, 3)

    def forward(self, X):
        Q = self.split_heads(self.W_q(X))
        K = self.split_heads(self.W_k(X))
        V = self.split_heads(self.W_v(X))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        Y = torch.matmul(attn, V)
        Y = Y.permute(2, 0, 1, 3).contiguous().view(X.shape)
        return self.W_o(Y), attn


X = torch.randn(6, 2, 4)
mha = MultiHeadSelfAttention(d_model=4, num_heads=2)
Y, attn = mha(X)
print("output shape =", Y.shape)
print("attention weight shape =", attn.shape)

output shape = torch.Size([6, 2, 4])
attention weight shape = torch.Size([2, 2, 6, 6])
